# <center> H&E Image CNN Classifier </center>

## Google Colab / GPU setup

In [ ]:
# Detect Colab if present
try:
    from google.colab import drive
    COLAB = True
    print('Note: using Google Colab')
except:
    print('Note: not using Google Colab')
    COLAB = False

# Use GPU or MPS (Apple) if available
import torch
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## H&E Images Dataset

In [ ]:
import numpy as np
import os
from patchify import patchify
from PIL import Image
from torch.utils.data import Dataset


class HEDataset(Dataset):
    def __init__(self, base_dir, patch_size, step):
        self.base_dir = base_dir
        self.patch_size = patch_size
        self.step = step
        self.samples = []
        self._prepare_dataset()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        with Image.open(file_path) as img:
            #img = img.resize((1576, 1576))
            img_array = np.array(img)
        
        patches = self.create_patches(img_array)
        filename = os.path.basename(file_path)
        core_id = os.path.splitext(filename)[0]
        patches_info = [(patch, label, core_id) for patch, _ in patches]
        
        patches, labels, filenames = zip(*patches_info)
        return np.array(patches), labels[0], filenames[0]

    def create_patches(self, img_array):
        patches = patchify(img_array, (self.patch_size, self.patch_size, 3), step=self.step)
        img_patches = []
        for i in range(patches.shape[0]):
            for j in range(patches.shape[1]):
                if not self._is_background_patch(patches[i, j, 0]):
                    img_patches.append((patches[i, j, 0], (i, j)))
        return img_patches

    def _is_background_patch(self, patch, threshold=0.8):
        return np.mean(patch > 240) > threshold
        #return np.mean(np.all(patch > 240, axis=-1)) > threshold

    def _prepare_dataset(self):
        label_dirs = {'malignant': 1, 'non_malignant': 0}
        for label_dir, label in label_dirs.items():
            dir_path = os.path.join(self.base_dir, label_dir)
            files = [os.path.join(dir_path, f) for f in os.listdir(dir_path) if f.endswith('.tif')]
            for file in files:
                self.samples.append((file, label))


full_dataset = HEDataset('../dataset/he_core_all_images/', 224, 224)

print('Number of cores in dataset:', len(full_dataset))

# Check first few cores in dataset
for i in range(3):
    patches, label, source = full_dataset[i]
    print(f'Core {source} has {len(patches)} patches, Label: {label}')


In [ ]:
import matplotlib.pyplot as plt

patch_counts = [len(full_dataset[i][0]) for i in range(len(full_dataset))]
plt.hist(patch_counts, bins=20)
plt.title('Distribution of patches per core')
plt.xlabel('Number of Patches')
plt.ylabel('Frequency')
plt.show()

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms

BATCH_SIZE = 4


def get_train_transform():
    train_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_transform

def get_test_transform():
    test_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return test_transform


# Applies augmentations to dataset (necessary to ensure training subset is augmented but not validation subset in cross-validation later)
class WrapperDataset(Dataset):
    def __init__(self, base_dataset, transform=None):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        patches, label, file = self.base_dataset[idx]
        
        if self.transform:
            patches = [self.transform(patch) for patch in patches]

        return patches, label, file


augmented_dataset = WrapperDataset(base_dataset=full_dataset, transform=get_train_transform())
dataloader = DataLoader(augmented_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda x: x)

# Check first batch
for batch in dataloader:
    print(f'Batch size (number of cores/bags): {len(batch)}')
    patches, labels, sources = zip(*batch)
    print(f'Number of patches in the first core/bag: {len(patches[0])}')
    print(f'Label of the first core/bag: {labels[0]}')
    print(f'Source of the first core/bag: {sources[0]}')
    break  

In [ ]:
def display_patch(img):
    # Unnormalize
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img.numpy().transpose((1, 2, 0)) 
    img = np.clip(img * std + mean, 0, 1)
    plt.imshow(img)
    plt.axis('off')
    plt.title('Example patch from a core')
    plt.show()

# Display random patch
for batch in dataloader:
    patches, label, source = batch[0]  
    display_patch(patches[15] )  
    break  